# Analysis of Global Properties (run level)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from exp_utils import REPO_ROOT
from exp_plot_utils import (
    _configure_fonts,
    plot_emmeans_grouped_bar,
    SCENARIO_MAP,
    SCENARIO_COLOR,
    SCENARIO_ORDER,
    OKABE_ITO,
    GRAPH_COLOR,
    GRAPH_MAP,
)

# exp_utils already puts REPO_ROOT on sys.path; r_utils lives in notebooks/.
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
from r_utils import (
    fit_lmer_full,
    fit_lmer_with_contrasts,
    build_contrast_summary,
    save_model_results,
    fit_family_direct_contrasts,
)

_configure_fonts()


In [ ]:
data_path = REPO_ROOT / "data" / "outputs" / "aggregated_expanded"
result_path = REPO_ROOT / "data" / "analysis" / "aggregated_expanded"
agg_path = result_path / "aggregated" / "runs"
cts_path = result_path / "contrasts" / "runs"
for path in (agg_path, cts_path):
    path.mkdir(parents=True, exist_ok=True)

# pandas keeps the category dtypes and orders E1 wrote (run_scenario, graph_subtype, run_id, ...).
dft = pd.read_parquet(data_path / "run_level_data.parquet")
dft.sample(10, random_state=0)


## Aggregates

In [ ]:
COL_RUN_PRETTY_NAMES = {
    "pop_consensus_change": "Consensus Change",
}

In [ ]:
for var_name in COL_RUN_PRETTY_NAMES:
    res = fit_lmer_full(
        df=dft,
        specification=f"""
            {var_name} ~ run_scenario * graph_family +
            (1 | statement_id) +
            (1 | graph_family:graph_seed)
        """,
        emm_formula="~run_scenario * graph_family",
        reml=True,
    )
    res_pooled = fit_lmer_full(
        df=dft,
        specification=f"""
            {var_name} ~ run_scenario +
            (1 | statement_id) +
            (1 | graph_family:graph_seed)
        """,
        emm_formula="~run_scenario",
        reml=True,
    )
    save_model_results(res, f"{var_name}_scenario_family", agg_path)
    save_model_results(res_pooled, f"{var_name}_scenario", agg_path)
    print(f"Finished fitting model for {var_name} with run_scenario * graph_family")
    plot_emmeans_grouped_bar(
        emm_df=res["emm"],
        legend_title="Network Type",
        group_col="graph_family",
        group_colors=GRAPH_COLOR,
        group_labels=GRAPH_MAP,
        group_order=GRAPH_COLOR.keys(),
        y_label=f"{COL_RUN_PRETTY_NAMES.get(var_name, var_name)}",
        title="",
        figsize=(4.5, 2.75),
        show=True,
        out_fig=agg_path / f"{var_name}_grouped_bar.pdf",
    )

## Contrast

In [ ]:
col_list = [
    "pop_consensus_T",
    "pop_consensus_change",
]
GROUP_VAR = "graph_family"


RUN_RANDOM_EFFECTS = (
    "(1 | statement_id) + (1 | run_scenario:statement_id)"
    " + (1 | graph_id) + (1 | run_scenario:graph_id)"
)


# Run the contrast pipeline first, collecting results into a dict
results_dict = {}
for col in col_list:
    results_dict[col] = fit_lmer_with_contrasts(
        dft,
        outcome=col,
        random_effects=RUN_RANDOM_EFFECTS,
        scenario_levels=SCENARIO_ORDER,
        spec_extra=f"* {GROUP_VAR}",
        group_var=GROUP_VAR,
        bootstrap=False,
    )
    print(f"contrasts done: {col}")

# Holm-adjusted p (p_holm) covers the family x contrast interaction tests only.
summary = build_contrast_summary(col_list, results_dict, group_var=GROUP_VAR)

summary.to_csv(cts_path / "contrast_summary_runs.csv", index=False)
display(summary[summary.contrast_type == "marginal"].round(4))